In [1]:
import logging
import os

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

from napistu_torch.load.constants import FM_DEFS
from napistu_torch.load.foundation_models import FoundationModel

from scGPT import process_scgpt, SCGPT_DEFS
from analysis_utils import (
    compute_attention_from_weights
)

/opt/homebrew/Caskroom/miniforge/base/envs/scgpt/lib/python3.11/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/opt/homebrew/Caskroom/miniforge/base/envs/scgpt/lib/python3.11/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/opt/homebrew/Caskroom/miniforge/base/envs/scgpt/lib/python3.11/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/opt/homebrew/Caskroom/miniforge/base/envs/scgpt/lib/python3.11/site-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TO

In [2]:
# Configuration
DATA_DIR = "data"
OUTPUT_DIR = "output"

MODEL_PATH = os.path.join(DATA_DIR, "scGPT_bc")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
process_scgpt(MODEL_PATH, OUTPUT_DIR)

scGPT - INFO - Extracting: scGPT
scGPT - INFO - 
1. Downloading/loading gene annotations...
scGPT - INFO -    Loaded 60664 gene annotations
scGPT - INFO - 2. Loading scGPT model...
Resume model from data/scGPT_bc/best_model.pt, the model args will override the config data/scGPT_bc/args.json.
Loading params encoder.embedding.weight with shape torch.Size([60697, 512])
Loading params encoder.enc_norm.weight with shape torch.Size([512])
Loading params encoder.enc_norm.bias with shape torch.Size([512])
Loading params value_encoder.linear1.weight with shape torch.Size([512, 1])
Loading params value_encoder.linear1.bias with shape torch.Size([512])
Loading params value_encoder.linear2.weight with shape torch.Size([512, 512])
Loading params value_encoder.linear2.bias with shape torch.Size([512])
Loading params value_encoder.norm.weight with shape torch.Size([512])
Loading params value_encoder.norm.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.0.self_attn.out_proj.

INFO:napistu_torch.load.foundation_models:Saving weights to output/scGPT_weights.npz
INFO:napistu_torch.load.foundation_models:Saving metadata to output/scGPT_metadata.json
INFO:napistu_torch.load.foundation_models:Successfully saved all results


scGPT - INFO -    Successfully saved all results!


In [4]:
# Load FoundationModel
foundation_model = FoundationModel.load(OUTPUT_DIR, SCGPT_DEFS.MODEL_NAME)

GENES_OF_INTEREST = foundation_model.gene_annotations[FM_DEFS.VOCAB_NAME].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in foundation_model.ordered_vocabulary]

# Compute attention on demand
layer_11 = foundation_model.weights.attention_layers[11]
layer_11_attn = compute_attention_from_weights(
    foundation_model.weights.gene_embedding[GENE_MASK,:],
    layer_11.W_q,
    layer_11.W_k
)

INFO:napistu_torch.load.foundation_models:Loading weights (scGPT_weights.npz) and metadata (  scGPT_metadata.json) from output_dir (output)
INFO:napistu_torch.load.foundation_models:Successfully loaded all results
